# 01 — Model selection for the Luganda evaluation

This notebook selects **one** small open model (1B–4B parameters) for the Luganda minimal-pair evaluation.

**Method.** Each candidate model gets the same input:

1. **10 Luganda prompts** from `reuben256/alpaca-luganda-cleaned`. The model writes an answer. You rate the fluency of each answer (blind).
2. **200 Luganda reference texts** from the same dataset (different rows). The notebook computes the log-probability of each text. This is the same score as Test A, at text level.

**Metrics.**

| Metric | Source | Better |
|---|---|---|
| Human fluency (1–5) | Your blind ratings | Higher |
| Bits per byte (BPB) on reference texts | Log-probability. BPB does not depend on the tokenizer, so you can compare models | Lower |
| Luganda share of answers | GlotLID language detector (`lug_Latn`) | Higher |
| chrF++ against the reference answer | sacrebleu | Higher |
| Tokens per word (fertility) | Tokenizer | Information only (Question 3) |

**Decision rule.** The main criterion is human fluency. BPB breaks a tie. The other metrics support the decision.

**Before you start** — see the checklist in `README.md`:
- Colab runtime with a GPU (L4 or A100 recommended; T4 works).
- Colab secret `HF_TOKEN`.
- Accepted licenses for the Gemma 3 and Llama 3.2 repositories on Hugging Face.

Run the cells from top to bottom. The model loop saves the results of each model immediately. If the runtime stops, run the notebook again: the loop skips models that have results.

## 1. Install packages

In [ ]:
%pip install -q "transformers==4.57.1" accelerate datasets sacrebleu huggingface_hub pandas fasttext-numpy2-wheel

## 2. Settings

Change these values if necessary. `BUCKET_URI` tells the notebook where to save all prompts, raw outputs and labels:

- `"gs://<bucket>/<path>"` — Google Cloud Storage bucket
- `"hf://datasets/<user>/<repo>/<path>"` — Hugging Face dataset repository (private, created if missing)
- `"/content/drive/MyDrive/<path>"` — Google Drive folder
- `""` — local Colab disk only (the files are lost when the runtime stops)

In [ ]:
DATASET_ID = "reuben256/alpaca-luganda-cleaned"
DATASET_SPLIT = "train"
SEED = 42

N_GEN_PROMPTS = 10      # prompts for generation and human rating (plan: 10)
N_SCORE_TEXTS = 200     # reference texts for the log-probability score
MAX_NEW_TOKENS = 256
SKIP_ROWS = []          # dataset row indices to exclude (for example, bad prompts)

OUTPUT_DIR = "/content/outputs/01_model_selection"
BUCKET_URI = ""         # see the list above
FORCE_RERUN = False     # True = run models again even if results exist
DELETE_WEIGHTS_AFTER_RUN = True  # remove each model from the HF cache after its run (saves disk)

# Candidate models (key -> Hugging Face repository).
# Add or remove entries after you look at the trending list in section 4.
CANDIDATES = {
    "qwen3-4b":    "Qwen/Qwen3-4B",
    "gemma3-4b":   "google/gemma-3-4b-it",
    "llama3.2-3b": "meta-llama/Llama-3.2-3B-Instruct",
    "smollm3-3b":  "HuggingFaceTB/SmolLM3-3B",
}

# The same system prompt for all models. It is in English so that every model understands it.
SYSTEM_PROMPT = (
    "You are a helpful assistant. Always answer in Luganda only. "
    "Do not use English. Keep the answer short and clear."
)

## 3. Log in to Hugging Face and check the GPU

In [ ]:
import os, json, math, time, re, gc, random, shutil, subprocess
from pathlib import Path

import torch
import transformers
from huggingface_hub import login, whoami, HfApi, hf_hub_download

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Colab secret HF_TOKEN not read:", e)

login(token=os.environ.get("HF_TOKEN"), add_to_git_credential=False)
print("Logged in as:", whoami()["name"])

assert torch.cuda.is_available(), "No GPU. Select Runtime > Change runtime type > GPU."
GPU_NAME = torch.cuda.get_device_name(0)
# bf16 needs Ampere or newer (L4, A100). T4 uses fp16.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(f"GPU: {GPU_NAME} | dtype: {DTYPE} | transformers {transformers.__version__} | torch {torch.__version__}")
if DTYPE == torch.float16:
    print("WARNING: fp16 on this GPU. Gemma 3 can give NaN scores in fp16. Use an L4 or A100 runtime if possible.")

out = Path(OUTPUT_DIR)
(out / "raw").mkdir(parents=True, exist_ok=True)

### 3.1 Check access to each model

Gemma 3 and Llama 3.2 are gated. If this cell shows `NO ACCESS`, open the model page on Hugging Face, accept the license, wait for approval, and run the cell again.

In [ ]:
api = HfApi()
access_ok = {}
for key, repo in CANDIDATES.items():
    try:
        hf_hub_download(repo, "config.json")
        access_ok[key] = True
        print(f"OK         {repo}")
    except Exception as e:
        access_ok[key] = False
        print(f"NO ACCESS  {repo}  ->  https://huggingface.co/{repo}  ({type(e).__name__})")

## 4. Look at the Hugging Face trending list (optional)

The plan says: check the trending list. This cell shows trending text-generation models with 1B–4B parameters. If you find a better candidate, add it to `CANDIDATES` in section 2 and run section 2 and 3 again.

In [ ]:
try:
    rows = []
    for m in api.list_models(pipeline_tag="text-generation", sort="trending_score",
                             limit=300, expand=["safetensors", "downloads"]):
        st = getattr(m, "safetensors", None)
        n = getattr(st, "total", None) if st else None
        if n and 1.0e9 <= n <= 4.6e9:
            rows.append((m.id, round(n / 1e9, 2), getattr(m, "downloads", None)))
    print(f"{'model':60s} {'params (B)':>10s} {'downloads':>12s}")
    for r in rows[:30]:
        print(f"{r[0]:60s} {r[1]:>10} {str(r[2]):>12s}")
except Exception as e:
    print("Trending list not available:", e)

## 5. Load the dataset and take the sample

The notebook finds the instruction, input and output columns automatically (Alpaca format). It uses a fixed seed, so the sample is the same each time. The 10 generation prompts and the 200 reference texts come from **different** rows.

The filters remove rows with code, many digits, or very short or very long text, because these rows do not test Luganda fluency.

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
print(ds)

def find_col(names, options):
    low = {c.lower(): c for c in names}
    for o in options:
        if o in low:
            return low[o]
    return None

COL_INSTR = find_col(ds.column_names, ["instruction", "prompt", "question", "instruction_lg", "lug_instruction"])
COL_INPUT = find_col(ds.column_names, ["input", "context", "input_lg", "lug_input"])
COL_OUTPUT = find_col(ds.column_names, ["output", "response", "answer", "output_lg", "lug_output"])
print("Columns used:", {"instruction": COL_INSTR, "input": COL_INPUT, "output": COL_OUTPUT})
assert COL_INSTR and COL_OUTPUT, f"Set COL_INSTR / COL_OUTPUT manually. Columns: {ds.column_names}"

CODE_MARKERS = ("```", "def ", "print(", "{", "}", "</", "SELECT ", "import ", "#include", "=>", "==")

def clean(s):
    return (s or "").strip()

def looks_bad(text):
    if any(m in text for m in CODE_MARKERS):
        return True
    if not text:
        return False
    digits = sum(ch.isdigit() for ch in text)
    return digits / len(text) > 0.08

def is_gen_candidate(r):
    ins, inp, outp = clean(r["instruction"]), clean(r["input"]), clean(r["output"])
    return (15 <= len(ins) <= 300 and len(inp) <= 400 and 40 <= len(outp) <= 800
            and not looks_bad(ins + " " + inp + " " + outp))

def is_score_candidate(r):
    outp = clean(r["output"])
    return 80 <= len(outp) <= 1000 and not looks_bad(outp)

rows = []
for i, r in enumerate(ds):
    if i in SKIP_ROWS:
        continue
    rows.append({"row": i,
                 "instruction": clean(r[COL_INSTR]),
                 "input": clean(r[COL_INPUT]) if COL_INPUT else "",
                 "output": clean(r[COL_OUTPUT])})

rng = random.Random(SEED)
rng.shuffle(rows)
gen_items = [r for r in rows if is_gen_candidate(r)][:N_GEN_PROMPTS]
used = {r["row"] for r in gen_items}
score_items = [r for r in rows if r["row"] not in used and is_score_candidate(r)][:N_SCORE_TEXTS]
print(f"Generation prompts: {len(gen_items)} | Reference texts: {len(score_items)}")

for k, r in enumerate(gen_items):
    r["id"] = f"P{k+1:02d}"
    r["prompt"] = r["instruction"] + (("\n\n" + r["input"]) if r["input"] else "")
for k, r in enumerate(score_items):
    r["id"] = f"S{k+1:03d}"

def write_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

write_jsonl(out / "gen_prompts.jsonl", gen_items)
write_jsonl(out / "score_texts.jsonl", score_items)

for r in gen_items:
    print(f"--- {r['id']} (row {r['row']})\n{r['prompt']}\n")

**Check the 10 prompts above.** If a prompt is not good Luganda, or it is not a fair test, add its row number to `SKIP_ROWS` in section 2. Then run section 2 and section 5 again.

## 6. Helper functions

- `score_text` gives the sum of `log P(token | previous tokens)` for a text. Each text starts after one start token (BOS) that is not scored. This is the same method as Test A.
- Bits per byte = `−sum_logprob / ln(2) / number of UTF-8 bytes`. Different tokenizers cut the same text into different numbers of tokens. BPB divides by bytes, so the value is fair across models.
- `generate` uses greedy decoding (no sampling), so the result is reproducible. Thinking mode is off for Qwen3 and SmolLM3.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(repo):
    tok = AutoTokenizer.from_pretrained(repo)
    kw = dict(dtype=DTYPE, device_map="cuda")
    try:
        model = AutoModelForCausalLM.from_pretrained(repo, **kw)
    except (ValueError, KeyError):
        # Gemma 3 4B is a vision-language checkpoint. Load the full model; we use text input only.
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(repo, **kw)
    model.eval()
    return tok, model

def start_token_id(tok):
    if tok.bos_token_id is not None:
        return tok.bos_token_id, tok.bos_token
    vocab = tok.get_vocab()
    for t in ("<|begin_of_text|>", "<|endoftext|>"):
        if t in vocab:
            return vocab[t], t
    return tok.eos_token_id, tok.eos_token

@torch.no_grad()
def score_text(model, tok, text):
    sid, _ = start_token_id(tok)
    ids = tok(text, add_special_tokens=False)["input_ids"]
    input_ids = torch.tensor([[sid] + ids], device=model.device)
    logits = model(input_ids=input_ids).logits[0, :-1].float()
    logp = torch.log_softmax(logits, dim=-1)
    tok_lp = logp.gather(1, input_ids[0, 1:, None]).squeeze(1)
    total = tok_lp.sum().item()
    n_bytes = len(text.encode("utf-8"))
    return {
        "n_tokens": len(ids),
        "n_words": len(text.split()),
        "n_bytes": n_bytes,
        "sum_logprob": total,
        "mean_logprob_per_token": total / len(ids),
        "bits_per_byte": -total / math.log(2) / n_bytes,
        "is_finite": bool(math.isfinite(total)),
    }

def chat_text(tok, user_text):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        # Some chat templates do not accept a system role.
        msgs = [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_text}]
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

THINK_RE = re.compile(r"<think>.*?</think>", flags=re.S)

@torch.no_grad()
def generate(model, tok, user_text):
    text = chat_text(tok, user_text)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out_ids = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None, top_p=None, top_k=None,
        repetition_penalty=1.05,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
    )
    new = out_ids[0, enc["input_ids"].shape[1]:]
    raw = tok.decode(new, skip_special_tokens=True)
    return {"chat_input": text, "raw_output": raw, "output": THINK_RE.sub("", raw).strip(),
            "n_new_tokens": int(new.shape[0]), "hit_max_tokens": int(new.shape[0]) >= MAX_NEW_TOKENS}

## 7. Run all candidate models

For each model: load it, score the 200 reference texts, generate answers for the 10 prompts, save the results, and free the GPU memory. Expect about 5–15 minutes for each model (most of the time is the download).

In [ ]:
from tqdm.auto import tqdm

run_info = {"gpu": GPU_NAME, "dtype": str(DTYPE), "transformers": transformers.__version__,
            "torch": torch.__version__, "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS,
            "system_prompt": SYSTEM_PROMPT, "dataset": DATASET_ID, "split": DATASET_SPLIT,
            "columns": {"instruction": COL_INSTR, "input": COL_INPUT, "output": COL_OUTPUT},
            "candidates": CANDIDATES, "models": {}}
info_path = out / "run_info.json"
if info_path.exists():
    run_info["models"] = json.loads(info_path.read_text()).get("models", {})

for key, repo in CANDIDATES.items():
    gen_path, score_path = out / "raw" / f"gen_{key}.jsonl", out / "raw" / f"score_{key}.jsonl"
    if gen_path.exists() and score_path.exists() and not FORCE_RERUN:
        print(f"[{key}] results exist, skip.")
        continue
    if not access_ok.get(key, False):
        print(f"[{key}] no access, skip. Accept the license and run section 3.1 again.")
        continue

    print(f"\n===== {key}: {repo} =====")
    t0 = time.time()
    tok, model = load_model(repo)
    sid, stok = start_token_id(tok)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"parameters: {n_params/1e9:.2f}B | start token: {stok!r}")

    scores = []
    for it in tqdm(score_items, desc="score"):
        s = score_text(model, tok, it["output"])
        scores.append({"id": it["id"], "row": it["row"], "model": key, **s})
    write_jsonl(score_path, scores)

    gens = []
    for it in tqdm(gen_items, desc="generate"):
        g = generate(model, tok, it["prompt"])
        gens.append({"id": it["id"], "row": it["row"], "model": key, "prompt": it["prompt"],
                     "reference": it["output"], **g})
    write_jsonl(gen_path, gens)

    run_info["models"][key] = {"repo": repo, "n_params": n_params, "start_token": stok,
                               "minutes": round((time.time() - t0) / 60, 1)}
    info_path.write_text(json.dumps(run_info, indent=2, ensure_ascii=False))

    print(f"[{key}] done in {run_info['models'][key]['minutes']} min. Example answer ({gens[0]['id']}):")
    print(gens[0]["output"][:500])

    del model, tok
    gc.collect(); torch.cuda.empty_cache()
    if DELETE_WEIGHTS_AFTER_RUN:
        from huggingface_hub import scan_cache_dir
        for r in scan_cache_dir().repos:
            if r.repo_id == repo:
                scan_cache_dir().delete_revisions(*[rev.commit_hash for rev in r.revisions]).execute()

## 8. Automatic metrics

- **Luganda share**: GlotLID labels each answer. The share is the fraction of answers labelled `lug_Latn`.
- **chrF++**: character overlap with the reference answer. One prompt has many good answers, so chrF++ is a weak signal. Use it as support only.

In [ ]:
import fasttext
from sacrebleu.metrics import CHRF

lid = fasttext.load_model(hf_hub_download("cis-lmu/glotlid", "model.bin"))
chrf = CHRF(word_order=2)

def lang_id(text):
    labels, probs = lid.predict(text.replace("\n", " ")[:2000] or " ", k=1)
    return labels[0].replace("__label__", ""), float(probs[0])

models_done = [k for k in CANDIDATES if (out / "raw" / f"gen_{k}.jsonl").exists()]
gen_all, score_all = [], []
for k in models_done:
    for g in read_jsonl(out / "raw" / f"gen_{k}.jsonl"):
        g["lang"], g["lang_prob"] = lang_id(g["output"])
        g["chrf"] = chrf.sentence_score(g["output"], [g["reference"]]).score
        gen_all.append(g)
    score_all += read_jsonl(out / "raw" / f"score_{k}.jsonl")

gen_df, score_df = pd.DataFrame(gen_all), pd.DataFrame(score_all)
gen_df.drop(columns=["chat_input"]).to_csv(out / "generations_with_auto_metrics.csv", index=False)

if not score_df["is_finite"].all():
    print("WARNING: some scores are not finite (NaN/inf). Check the dtype (section 3).")
    print(score_df.loc[~score_df["is_finite"], ["model", "id"]].value_counts("model"))

auto = pd.DataFrame({
    "bits_per_byte": score_df[score_df.is_finite].groupby("model")["bits_per_byte"].mean(),
    # corpus-level BPB: total bits / total bytes
    "corpus_bpb": score_df[score_df.is_finite].groupby("model").apply(
        lambda d: -d.sum_logprob.sum() / math.log(2) / d.n_bytes.sum()),
    "tokens_per_word": score_df.groupby("model").apply(lambda d: d.n_tokens.sum() / d.n_words.sum()),
    "luganda_share": gen_df.groupby("model")["lang"].apply(lambda s: (s == "lug_Latn").mean()),
    "chrf": gen_df.groupby("model")["chrf"].mean(),
    "hit_max_tokens": gen_df.groupby("model")["hit_max_tokens"].mean(),
}).loc[models_done]
auto.round(3)

In [ ]:
# Languages that GlotLID found in the answers of each model
gen_df.groupby("model")["lang"].value_counts().unstack(fill_value=0)

## 9. Human fluency rating (blind)

This cell writes `rating_sheet.csv`. The sheet hides the model names and shuffles the order, so your rating is not biased. The key is in a separate file (`rating_key.csv`). Do not open the key before you finish.

**How to rate.** Open the CSV in Google Sheets or Excel. Fill the column `fluency` (1–5) for each row. The columns `adequacy` (1–5) and `notes` are optional.

| Score | Fluency |
|---|---|
| 5 | Natural Luganda, like a native speaker. No grammar errors. |
| 4 | Small errors (for example, one wrong agreement). Meaning is clear. |
| 3 | Several errors. Meaning is mostly clear. |
| 2 | Many errors, or a mix of Luganda with English or another language. |
| 1 | Not Luganda, or not understandable. |

`adequacy`: does the answer do what the prompt asks? (5 = fully, 1 = not at all.)

In `notes`, write the error types that you see (noun-class agreement, tense/aspect, negation, subject/object marker). These notes help you write the minimal pairs later.

Save the file as CSV with the same columns. Then run section 10.

In [ ]:
sheet = gen_df[["id", "model", "prompt", "output"]].copy()
rng_sheet = random.Random(SEED + 1)
codes = {m: f"M{i+1}" for i, m in enumerate(rng_sheet.sample(models_done, len(models_done)))}
sheet["system"] = sheet["model"].map(codes)
sheet = sheet.sample(frac=1.0, random_state=SEED + 1).reset_index(drop=True)
sheet.insert(0, "item", [f"R{i+1:03d}" for i in range(len(sheet))])
sheet[["item", "model", "system"]].to_csv(out / "rating_key.csv", index=False)
sheet = sheet.drop(columns=["model"])
sheet["fluency"] = ""
sheet["adequacy"] = ""
sheet["notes"] = ""
sheet.to_csv(out / "rating_sheet.csv", index=False)
print(f"Wrote {len(sheet)} rows to {out/'rating_sheet.csv'}")

try:
    from google.colab import files
    files.download(str(out / "rating_sheet.csv"))
except Exception:
    pass

## 10. Load your ratings

Upload the filled sheet. Save it with the name `rating_sheet_filled.csv`, or change `RATINGS_PATH`.

In [ ]:
RATINGS_PATH = out / "rating_sheet_filled.csv"

if not RATINGS_PATH.exists():
    try:
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        RATINGS_PATH.write_bytes(up[name])
    except Exception as e:
        print("Upload the file to", RATINGS_PATH, "|", e)

ratings = pd.read_csv(RATINGS_PATH)
key = pd.read_csv(out / "rating_key.csv")
ratings = ratings.merge(key[["item", "model"]], on="item", how="left")
ratings["fluency"] = pd.to_numeric(ratings["fluency"], errors="coerce")
ratings["adequacy"] = pd.to_numeric(ratings["adequacy"], errors="coerce")
missing = ratings["fluency"].isna().sum()
print(f"Rated rows: {len(ratings) - missing}/{len(ratings)}")
if missing:
    print("WARNING: some rows have no fluency score.")

human = ratings.groupby("model").agg(
    fluency_mean=("fluency", "mean"), fluency_sd=("fluency", "std"),
    adequacy_mean=("adequacy", "mean"), n_rated=("fluency", "count"))
human.round(2)

## 11. Final ranking and decision

Sort order: human fluency (higher is better), then corpus BPB (lower is better). If there are no ratings yet, the cell uses an automatic rank only and prints a warning. Do not make the final choice without human ratings.

In [ ]:
table = auto.copy()
have_human = "human" in globals() and human["n_rated"].sum() > 0
if have_human:
    table = table.join(human, how="left")
    table = table.sort_values(["fluency_mean", "corpus_bpb"], ascending=[False, True])
else:
    print("WARNING: no human ratings. This ranking uses automatic metrics only.")
    r = (table["corpus_bpb"].rank() + table["luganda_share"].rank(ascending=False)
         + table["chrf"].rank(ascending=False))
    table = table.assign(auto_rank=r).sort_values("auto_rank")

table["repo"] = [CANDIDATES[m] for m in table.index]
table.to_csv(out / "model_selection_summary.csv")
display(table.round(3))

best = table.index[0]
b = table.loc[best]
others = table.drop(index=best)
runner = others.index[0] if len(others) else None

parts = [f"We selected {CANDIDATES[best]} for the Luganda evaluation."]
if have_human:
    parts.append(f"In a blind rating of {int(b.n_rated)} answers to the same {len(gen_items)} Luganda prompts, "
                 f"it had the highest mean fluency ({b.fluency_mean:.2f}/5"
                 + (f", against {table.loc[runner].fluency_mean:.2f} for {CANDIDATES[runner]})." if runner else ")."))
parts.append(f"On {len(score_items)} Luganda reference texts it scored {b.corpus_bpb:.3f} bits per byte"
             + (f" ({table.loc[runner].corpus_bpb:.3f} for {CANDIDATES[runner]})" if runner else "")
             + ", which shows how well the model predicts real Luganda text; this score uses the same "
               "log-probability as Test A.")
parts.append(f"GlotLID identified {b.luganda_share:.0%} of its answers as Luganda, "
             f"and its tokenizer used {b.tokens_per_word:.2f} tokens per word.")
parts.append("[Add your own observation here: for example, the error types you saw most often in the answers.]")
paragraph = " ".join(parts)
(out / "model_choice_paragraph.md").write_text(paragraph + "\n", encoding="utf-8")
print("\nDRAFT PARAGRAPH (edit before you use it):\n")
print(paragraph)

## 12. Save everything to the bucket

This cell copies the full output folder (prompts, raw outputs, scores, rating sheet, your ratings, summary, run information) to `BUCKET_URI`.

In [ ]:
def sync_to_bucket(local_dir, uri):
    local_dir = str(local_dir)
    if not uri:
        print("BUCKET_URI is empty. Files are only on the local disk:", local_dir)
    elif uri.startswith("gs://"):
        from google.colab import auth
        auth.authenticate_user()
        subprocess.run(["gsutil", "-m", "rsync", "-r", local_dir, uri.rstrip("/")], check=True)
    elif uri.startswith("hf://datasets/"):
        parts = uri[len("hf://datasets/"):].strip("/").split("/")
        repo_id, sub = "/".join(parts[:2]), "/".join(parts[2:])
        api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)
        api.upload_folder(folder_path=local_dir, repo_id=repo_id, repo_type="dataset",
                          path_in_repo=sub or None, commit_message="01_model_selection outputs")
    elif uri.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copytree(local_dir, uri, dirs_exist_ok=True)
    else:
        raise ValueError(f"Unknown BUCKET_URI: {uri}")
    print("Saved to:", uri or local_dir)

sync_to_bucket(out, BUCKET_URI)
for p in sorted(out.rglob("*")):
    if p.is_file():
        print(p.relative_to(out))